# Building a house as minimum-cost network flow

Use the same task durations and precedence relations as [11-House.ipynb](11-House.ipynb). Tasks `:a` and `:x` are the existing zero-duration start and finish nodes.

Send one unit of flow from `:a` to `:x`. Each precedence arc $(i,j)$ has cost $c_{ij}=-d_j$, the negative duration of its destination task, as in the lecture. Set $b_a=1$, $b_x=-1$, and all other net supplies to zero, with $\ell_{ij}=0$ and $u_{ij}=\infty$.

The precedence graph is acyclic, so these negative costs give a bounded longest-path problem. The minimum project completion time is **the negative of the minimum network cost**. Positive flow identifies a critical path; tasks outside that path still have to be completed and may run in parallel.

Open the course repository root in VS Code, select the Julia 1.12 kernel with the course environment, and choose **Run All**. All required packages are already included, and no external data files are needed.


## Data


In [ ]:
# this array stores the task names (:a, :b, ..., :x)
tasks = []
for i = 'a':'x'
    push!(tasks, Symbol(i))    # string(sym) converts back to a string, i.e. string(:hello) returns "hello"
end

# this dictionary stores the project durations
dur = [0, 4, 2, 4, 6, 1, 2, 3, 2, 4, 10, 3, 1, 2, 3, 2, 1, 1, 2, 3, 1, 2, 5, 0]
duration = Dict(zip(tasks,dur))

# this dictionary stores the projects that a given project depends on (ancestors)
pre = ( [], [:a], [:b], [:c], [:d], [:c], [:f], [:f], [:d], [:d,:g], [:i,:j,:h], [:k],
    [:l], [:l], [:l], [:e], [:p], [:c], [:o,:t], [:m,:n], [:t], [:q,:r], [:v], [:s,:u,:w])
pred = Dict(zip(tasks,pre));


# Convert the precedence graph into MCNF data.
nodes = tasks
arcs = [(i,j) for j in nodes for i in pred[j]]

# Duration of the destination task, negated to find a longest path by minimization.
c = Dict((i,j) => -duration[j] for (i,j) in arcs)

# Send one unit from the existing dummy start to the existing dummy finish.
b = Dict(i => 0 for i in nodes)
b[:a] = 1
b[:x] = -1

ℓ = Dict((i,j) => 0 for (i,j) in arcs)
u = Dict((i,j) => Inf for (i,j) in arcs)


## Model

This is the unchanged algebraic model from [12-mcnf.ipynb](12-mcnf.ipynb).


In [ ]:
using JuMP, HiGHS
import MathOptInterface as MOI

m = Model()

@variable(m, ℓ[i,j] <= x[(i,j) in arcs] <= u[i,j])

@objective(m, Min,
    sum(c[i,j]x[(i,j)] for (i,j) in arcs))

@constraint(m, flowbalance[i in nodes],
    sum(x[(i,j)] for j in nodes if (i,j) in arcs) -
    sum(x[(j,i)] for j in nodes if (j,i) in arcs)
    == b[i])

set_optimizer(m, HiGHS.Optimizer)
set_silent(m)
optimize!(m)


## Solution


In [ ]:
println("Status: ", termination_status(m))
if termination_status(m) != MOI.OPTIMAL || primal_status(m) != MOI.FEASIBLE_POINT
    error("No optimal primal solution: $(termination_status(m)), $(primal_status(m))")
end

println("Minimum network cost = ", round(objective_value(m), digits=1))
println("Minimum project completion time = ", round(-objective_value(m), digits=1), " days")
println("Critical-path arcs (positive flow):")
for (i,j) in arcs
    if value(x[(i,j)]) > 0.00001
        println(i, " -> ", j, ": flow = ", round(value(x[(i,j)]), digits=1))
    end
end
